# CPSC 368 – Group 4: YouTube Data Collection
## Phase 3 – Data Collection from ANES datasets

This notebook collects ANES data and refines into CSV files that can then be uploaded to SQL Oracle


In [36]:
import time
import csv
import json
import pandas as pd
import os
import numpy as np

from dotenv import load_dotenv

### Loading in Data

In [29]:
#2024 election study data:
usecols2024 = ['V240001', 'V200001', 'V160001_orig', 'V241177',  'V242125', 'V242126', 'V242577e'] 
df_anes_2024 = pd.read_csv('../../source_data/anes/anes_timeseries_2024_csv_20250808.csv',
                          usecols = usecols2024
                          )
df_anes_2024['election_year'] = 2024

#2020 election study data:
usecols2020 = ['V200001', 'V160001_orig', 'V201200', 'V202143', 'V202144', 'V202541e'] 
df_anes_2020 = pd.read_csv('../../source_data/anes/anes_timeseries_2020_csv_20220210.csv',
                          usecols = usecols2020)
df_anes_2020['election_year'] = 2020

In [30]:
# #2024 election study data:
display(df_anes_2024.head())

,V240001,V200001,V160001_orig,V241177,V242125,V242126,V242577e,election_year
0,140001,200015,401318,6,0,85,1,2024
1,140002,200022,300261,4,50,50,0,2024
2,140003,200039,400181,2,85,0,0,2024
3,140004,200046,300171,99,85,-9,1,2024
4,140005,200053,405145,4,0,60,1,2024


In [23]:
# #2020 election study data:
display(df_anes_2020.head())

,V200001,V160001_orig,V201200,V202143,V202144,V202541e,election_year
0,200015,401318,6,0,100,1,2020
1,200022,300261,4,15,15,1,2020
2,200039,400181,2,85,0,1,2020
3,200046,300171,3,100,60,1,2020
4,200053,405145,5,0,90,1,2020


### Cleaning Tables

In [37]:
# cleaning 2024
df_anes_2024_clean = df_anes_2024.rename(columns={
    'V240001': 'id_2024', # respondent's assigned id in 2024
    'V200001': 'id_2020', # respondent's assigned id in 2020
    'V160001_orig': 'id_2016', # respondent's assigned id in 2016
    'V241177': 'lib_con_scale', # self reported scale from very liberal to very conservative
    'V242125': 'democrat_thermometer', # how they feel about democrats from 0 - 100
    'V242126': 'republican_thermometer', # how they feel about republicans from 0 - 100
    'V242577e': 'youtube_use', # do they visit youtube as part of their social media   
})

# meanings of codes
mappings = {
    'V160001_orig':
    'lib_con_scale': {  
        1: 'Extremely Liberal',
        2: 'Liberal',
        3: 'Slightly liberal',
        4: 'Moderate',
        5: 'Slightly conservative',
        6: 'Conservative',
        7: 'Extremely conservative',
        9: 'Refused',
        99: 'DK'
    },
    'youtube_use': {
        -9: np.nan,
        -7: np.nan,
        -6: np.nan,
        -5: np.nan,
        -1: np.nan,
        0: 'No',
        1: 'Yes'
    }
}
        

# replace codes with readable values
def map_values(df, mappings):
    for col, mapping in mappings.items():
        if col in df.columns:
            df[col] = df[col].map(mapping)
    return df

numeric_vars = [
    'democrat_thermometer',
    'republican_thermometer',
]
df_anes_2024_clean[numeric_vars] = df_anes_2024_clean[numeric_vars].replace([-1, -5, -6, -7, -9, 998, 999], np.nan)

df_anes_2024_clean = map_values(df_anes_2024_clean, mappings)

In [39]:
# cleaning 2020
df_anes_2020_clean = df_anes_2020.rename(columns={
    'V200001': 'id_2020', # respondent's assigned id in 2020
    'V160001_orig': 'id_2016', # respondent's assigned id in 2016
    'V201200': 'lib_con_scale', # self reported scale from very liberal to very conservative
    'V202143': 'democrat_thermometer', # how they feel about democrats from 0 - 100
    'V202144': 'republican_thermometer', # how they feel about republicans from 0 - 100
    'V202541e': 'youtube_use', # do they visit youtube as part of their social media   
})

# meanings of codes
mappings = {
    'lib_con_scale': {  
        1: 'Extremely Liberal',
        2: 'Liberal',
        3: 'Slightly liberal',
        4: 'Moderate',
        5: 'Slightly conservative',
        6: 'Conservative',
        7: 'Extremely conservative',
        -8: 'DK',
        -9: 'Refused',
        99: 'DK'
    },
    'youtube_use': {
        -9: np.nan,
        -7: np.nan,
        -6: np.nan,
        -5: np.nan,
        0: 'No',
        1: 'Yes'
    }
}

numeric_vars = [
    'democrat_thermometer',
    'republican_thermometer',
]
df_anes_2020_clean[numeric_vars] = df_anes_2020_clean[numeric_vars].replace([-4, -5, -6, -7, -9], np.nan)

df_anes_2020_clean = map_values(df_anes_2020_clean, mappings)

### Creating a smaller sample that can fit on the database server

In [42]:
# 2024 data
df_anes_2024_clean_sample = df_anes_2024_clean.sample(n=500, random_state=2026)

# 2024 data
df_anes_2020_clean_sample = df_anes_2020_clean.sample(n=500, random_state=2026)

In [43]:
# #2024 election study data:
display(df_anes_2020_clean_sample.head())

,id_2020,id_2016,lib_con_scale,democrat_thermometer,republican_thermometer,youtube_use,election_year
113,201438,403152,Conservative,0.0,85.0,Yes,2020
8159,529105,-1,Moderate,100.0,0.0,Yes,2020
7035,452663,-1,DK,0.0,100.0,No,2020
566,207245,407024,Slightly liberal,70.0,0.0,Yes,2020
4784,356789,-1,Liberal,90.0,0.0,Yes,2020


In [44]:
# #2020 election study data:
display(df_anes_2020_clean_sample.head())

,id_2020,id_2016,lib_con_scale,democrat_thermometer,republican_thermometer,youtube_use,election_year
113,201438,403152,Conservative,0.0,85.0,Yes,2020
8159,529105,-1,Moderate,100.0,0.0,Yes,2020
7035,452663,-1,DK,0.0,100.0,No,2020
566,207245,407024,Slightly liberal,70.0,0.0,Yes,2020
4784,356789,-1,Liberal,90.0,0.0,Yes,2020
